In [1]:
import os

cache_root = "/scratch/gilbreth/abelde"

os.environ["HOME"]              = cache_root
os.environ["PIP_CACHE_DIR"]     = os.path.join(cache_root, ".cache", "pip")
os.environ["TORCH_HOME"]        = os.path.join(cache_root, ".cache", "torch")
os.environ["HF_HOME"]           = os.path.join(cache_root, ".cache", "huggingface")
os.environ["XDG_CACHE_HOME"]    = os.path.join(cache_root, ".cache")
os.environ["TMPDIR"]            = os.path.join(cache_root, "tmp")

os.makedirs(os.environ["PIP_CACHE_DIR"], exist_ok=True)
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["XDG_CACHE_HOME"], exist_ok=True)
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

print("Cache directories set under:", cache_root)

Cache directories set under: /scratch/gilbreth/abelde


In [2]:
import os
import sys
import time

repo_root = "/scratch/gilbreth/abelde/Vision_Graphics/3DGS/gaussian-splatting"
data_path = "/scratch/gilbreth/abelde/Vision_Graphics/3DGS/datasets/garden"
output_dir = os.path.join(repo_root, "output/debug_garden")

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Warming up torch. The first import can take around 1-2 minutes in this environment...")
t0 = time.time()
import torch
print(f"torch imported in {time.time() - t0:.1f}s")
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Warming up torch. The first import can take around 1-2 minutes in this environment...
torch imported in 148.8s
torch version: 2.1.2
CUDA available: True
GPU: NVIDIA A30


In [3]:
import importlib
import sys
from argparse import ArgumentParser

import train
importlib.reload(train)

from arguments import ModelParams, PipelineParams, OptimizationParams
from utils.general_utils import safe_state

sys.argv = [
    "train.py",
    "--source_path", data_path,
    "--model_path", output_dir,
    "--iterations", "10",
    "--save_iterations", "10",
    "--test_iterations", "-1",
    "--resolution", "8",
    "--data_device", "cpu",
    "--max_train_cameras", "2",
    "--max_test_cameras", "1",
    "--densify_until_iter", "0",
    "--disable_viewer",
]

parser = ArgumentParser(description="Training script parameters")
lp = ModelParams(parser)
op = OptimizationParams(parser)
pp = PipelineParams(parser)
parser.add_argument("--ip", type=str, default="127.0.0.1")
parser.add_argument("--port", type=int, default=6009)
parser.add_argument("--debug_from", type=int, default=-1)
parser.add_argument("--detect_anomaly", action="store_true", default=False)
parser.add_argument("--test_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--save_iterations", nargs="+", type=int, default=[7_000, 30_000])
parser.add_argument("--quiet", action="store_true")
parser.add_argument("--disable_viewer", action="store_true", default=False)
parser.add_argument("--checkpoint_iterations", nargs="+", type=int, default=[])
parser.add_argument("--start_checkpoint", type=str, default=None)

args = parser.parse_args(sys.argv[1:])
args.save_iterations.append(args.iterations)

print("Optimizing", args.model_path)
print("Debug argv:")
print(" ".join(sys.argv))

Optimizing /scratch/gilbreth/abelde/Vision_Graphics/3DGS/gaussian-splatting/output/debug_garden
Debug argv:
train.py --source_path /scratch/gilbreth/abelde/Vision_Graphics/3DGS/datasets/garden --model_path /scratch/gilbreth/abelde/Vision_Graphics/3DGS/gaussian-splatting/output/debug_garden --iterations 10 --save_iterations 10 --test_iterations -1 --resolution 8 --data_device cpu --max_train_cameras 2 --max_test_cameras 1 --densify_until_iter 0 --disable_viewer


In [4]:
safe_state(args.quiet)
torch.autograd.set_detect_anomaly(args.detect_anomaly)

print("Running minimal debug training...")
train.training(
    lp.extract(args),
    op.extract(args),
    pp.extract(args),
    args.test_iterations,
    args.save_iterations,
    args.checkpoint_iterations,
    args.start_checkpoint,
    args.debug_from,
)

print("Training completed successfully.")

Running minimal debug training... [20/03 15:33:29]
Output folder: /scratch/gilbreth/abelde/Vision_Graphics/3DGS/gaussian-splatting/output/debug_garden [20/03 15:34:12]
Reading camera 185/185 [20/03 16:46:15]


AttributeError: can't set attribute